# Lab 03: DataFrame Transformations & Actions

## Objectives
- Distinguish between transformations (lazy) and actions (eager)
- Use column operations: select, withColumn, filter, cast, alias
- Understand Spark's lazy evaluation and DAG construction
- Use common actions: show, count, collect, take, first

## Exam Domain
- **Developing DataFrame/DataSet API Applications — 30%**

## Estimates
- **Time:** ~35 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Transformations vs Actions](../assets/diagrams/lab-03-transformations-actions.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

taxi_df = spark.table("taxi_trips")

## A. Lazy Evaluation — Transformations Build a Plan

Transformations like `filter()`, `select()`, and `withColumn()` do NOT execute immediately. They build a **logical plan** (a DAG of operations). Nothing runs on the cluster until you call an **action**.

In [ ]:
# These transformations execute NOTHING — they only build a plan
filtered = taxi_df.filter(taxi_df.trip_distance > 10)
selected = filtered.select("PULocationID", "DOLocationID", "trip_distance", "total_amount")
renamed = selected.withColumnRenamed("PULocationID", "pickup_location")

# No job has run yet — verify in Spark UI
print(type(renamed))  # Still a DataFrame, not results

In [ ]:
# Now trigger an action — THIS is when Spark executes the plan
count = renamed.count()
print(f"Trips longer than 10 miles: {count:,}")

# show() is also an action
renamed.show(5)

## B. Column Selection and Projection

`select()` chooses which columns to include. There are several syntax styles — the exam expects you to know all of them.

In [ ]:
from pyspark.sql.functions import col

# Style 1: String column names
taxi_df.select("VendorID", "trip_distance", "total_amount").show(3)

# Style 2: Column objects with col()
taxi_df.select(col("VendorID"), col("trip_distance"), col("total_amount")).show(3)

# Style 3: DataFrame dot notation
taxi_df.select(taxi_df.VendorID, taxi_df.trip_distance, taxi_df.total_amount).show(3)

# Style 4: Column indexing
taxi_df.select(taxi_df["VendorID"], taxi_df["trip_distance"], taxi_df["total_amount"]).show(3)

## C. Adding and Modifying Columns

`withColumn()` adds a new column or replaces an existing one. Combined with `cast()` for type conversion and `alias()` for renaming expressions.

In [ ]:
from pyspark.sql.functions import col, round, when, lit

# Add a calculated column (handle divide-by-zero with when)
df_with_tip_pct = taxi_df.withColumn(
    "tip_percentage",
    when(col("total_amount") != 0,
         round((col("tip_amount") / col("total_amount")) * 100, 2)
    ).otherwise(0.0)
)
df_with_tip_pct.select("total_amount", "tip_amount", "tip_percentage").show(5)

In [ ]:
# Cast a column to a different type
df_cast = taxi_df.withColumn("passenger_count", col("passenger_count").cast("integer"))
df_cast.printSchema()

In [ ]:
# Conditional column with when/otherwise
df_category = taxi_df.withColumn(
    "trip_category",
    when(col("trip_distance") < 2, "short")
    .when(col("trip_distance") < 10, "medium")
    .otherwise("long")
)
df_category.select("trip_distance", "trip_category").show(10)

## D. Filtering Rows

`filter()` and `where()` are identical — use either. The exam may use both interchangeably.

In [ ]:
# filter and where are the same operation
by_filter = taxi_df.filter(col("trip_distance") > 20)
by_where = taxi_df.where(col("trip_distance") > 20)

print(f"filter count: {by_filter.count()}")
print(f"where count: {by_where.count()}")

In [ ]:
# Multiple conditions
expensive_long_trips = taxi_df.filter(
    (col("trip_distance") > 10) & (col("total_amount") > 50)
)
expensive_long_trips.select("trip_distance", "total_amount", "tip_amount").show(5)

# isin() for membership testing
cash_or_card = taxi_df.filter(col("payment_type").isin(1, 2))
print(f"Cash or card trips: {cash_or_card.count():,}")

# between() for range filtering
mid_distance = taxi_df.filter(col("trip_distance").between(5, 15))
print(f"Mid-distance trips (5-15 miles): {mid_distance.count():,}")

### SQL Alternative

In [ ]:
# SQL equivalent of filtering
taxi_df.createOrReplaceTempView("taxi_view")

spark.sql("""
    SELECT trip_distance, total_amount, tip_amount
    FROM taxi_view
    WHERE trip_distance > 10 AND total_amount > 50
    LIMIT 5
""").show()

## E. Sorting and Deduplication

`orderBy()` / `sort()` sorts the DataFrame. `distinct()` and `dropDuplicates()` remove duplicates.

In [ ]:
# Sort — orderBy and sort are identical
taxi_df.orderBy(col("total_amount").desc()).select("total_amount", "trip_distance").show(5)

# Sort by multiple columns
taxi_df.sort(col("PULocationID").asc(), col("total_amount").desc()).show(5)

In [ ]:
# distinct() removes fully duplicate rows
locations = taxi_df.select("PULocationID").distinct()
print(f"Unique pickup locations: {locations.count()}")

# dropDuplicates() removes duplicates based on specific columns
unique_routes = taxi_df.dropDuplicates(["PULocationID", "DOLocationID"])
print(f"Unique routes: {unique_routes.count():,}")

## F. Actions — Triggering Execution

Actions return results to the driver or write data. Each action triggers a complete job execution.

| Action | Returns | Notes |
|--------|---------|-------|
| `show(n)` | None (prints to console) | Default n=20 |
| `count()` | int | Full table scan |
| `collect()` | List[Row] | All data to driver — dangerous on large data |
| `take(n)` | List[Row] | First n rows |
| `first()` | Row | Same as take(1)[0] |
| `head(n)` | List[Row] | Same as take(n) |
| `toPandas()` | pandas DataFrame | All data to driver — dangerous on large data |

In [ ]:
# Demonstrate actions
small_df = taxi_df.select("VendorID", "trip_distance").limit(5)

# collect() — returns all rows as a list of Row objects
rows = small_df.collect()
print(f"collect() returns: {type(rows)}")
for row in rows:
    print(f"  VendorID={row.VendorID}, distance={row.trip_distance}")

# first() — returns a single Row
first_row = small_df.first()
print(f"\nfirst() returns: {first_row}")

# take(n) — returns n rows
taken = small_df.take(3)
print(f"\ntake(3) returns {len(taken)} rows")

> **Exam Tip:** `collect()` and `toPandas()` bring ALL data to the driver. On large datasets, this will cause an OutOfMemoryError. Always `limit()` first or use `take(n)`. The exam tests whether you understand this risk.

## Exam-Style Review

**Q1.** Which of the following is a transformation (lazy), not an action?
- A) `count()`
- B) `show()`
- C) `filter()`
- D) `collect()`

**Q2.** What does `df.filter(col("x") > 5)` return?
- A) A list of rows where x > 5
- B) An integer count of matching rows
- C) A new DataFrame with the filter added to the logical plan
- D) Nothing — it modifies df in place

**Q3.** What is the risk of calling `df.collect()` on a 100GB DataFrame?
- A) It will be slow but work fine
- B) It will trigger a full shuffle
- C) It will cause an OutOfMemoryError on the driver
- D) It will only return the first 20 rows

**Q4.** Are `filter()` and `where()` different operations?
- A) Yes — `filter()` is for column expressions, `where()` is for SQL strings
- B) Yes — `where()` is lazily evaluated, `filter()` is eager
- C) No — they are identical and interchangeable
- D) Yes — `filter()` is deprecated in favor of `where()`

**Q5.** What does `withColumn("x", col("y") + 1)` do if column "x" already exists?
- A) Throws an error
- B) Creates a second column named "x"
- C) Replaces the existing column "x" with the new expression
- D) Appends the column at the end

### Answers
- **Q1: C** — `filter()` is a transformation. It builds a plan but does not execute until an action is called.
- **Q2: C** — Transformations return a new DataFrame. They do not execute or modify the original.
- **Q3: C** — `collect()` brings all data to the driver's memory, causing OutOfMemoryError on large datasets.
- **Q4: C** — They are aliases for the same operation.
- **Q5: C** — `withColumn()` replaces a column if one with the same name already exists.

## Key Takeaways
- Transformations are lazy (build a plan); actions are eager (trigger execution)
- Know all column selection styles: string, `col()`, dot notation, bracket notation
- `filter()` and `where()` are identical
- `withColumn()` replaces an existing column if the name matches
- Never `collect()` or `toPandas()` on large data without `limit()` first

**Next:** [Lab 04 — Aggregations & Grouping](04-aggregations-grouping.ipynb)

In [ ]:
# Cleanup
spark.catalog.dropTempView("taxi_view")
print("Lab 03 cleanup complete.")